1. Set up Patch Dan Muat Model Yolo V8 best

In [49]:
import io
import cv2
import numpy as np
from pathlib import Path
from flask import Flask, request, jsonify
from ultralytics import YOLO

# Setup Root Directory & Load Model
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_PATH = ROOT / "runs" / "wastify_yolov8n" / "weights" / "best.pt"
if not MODEL_PATH.exists():
    MODEL_PATH = ROOT / "runs" / "detect" / "wastify_yolov8n" / "weights" / "best.pt"

print(f"[INFO] Memuat Model Terbaik dari: {MODEL_PATH}")
model = YOLO(str(MODEL_PATH))

[INFO] Memuat Model Terbaik dari: d:\project semester 5\TUGAS AKHIR\WASTIFY_PCV\runs\wastify_yolov8n\weights\best.pt


2. Inisialisasi Fuzzy pada Api Backend

In [50]:
import skfuzzy as fuzz
from skfuzzy import control as ctrl

confidence = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'confidence')
area_ratio = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'area_ratio')
capacity = ctrl.Consequent(np.arange(0, 101, 1), 'capacity')

confidence['low'] = fuzz.trimf(confidence.universe, [0, 0, 0.5])
confidence['medium'] = fuzz.trimf(confidence.universe, [0.3, 0.6, 0.8])
confidence['high'] = fuzz.trimf(confidence.universe, [0.6, 1.0, 1.0])

area_ratio['small'] = fuzz.trimf(area_ratio.universe, [0, 0, 0.35])
area_ratio['medium'] = fuzz.trimf(area_ratio.universe, [0.25, 0.5, 0.75])
area_ratio['large'] = fuzz.trimf(area_ratio.universe, [0.6, 1.0, 1.0])

capacity['aman'] = fuzz.trimf(capacity.universe, [0, 0, 45])
capacity['hampir_penuh'] = fuzz.trimf(capacity.universe, [35, 60, 80])
capacity['penuh'] = fuzz.trimf(capacity.universe, [70, 100, 100])

rule1 = ctrl.Rule(area_ratio['small'], capacity['aman'])
rule2 = ctrl.Rule(area_ratio['medium'] & confidence['high'], capacity['hampir_penuh'])
rule3 = ctrl.Rule(area_ratio['medium'] & ~confidence['high'], capacity['aman'])
rule4 = ctrl.Rule(area_ratio['large'] & (confidence['medium'] | confidence['high']), capacity['penuh'])

capacity_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4])

def compute_fuzzy(conf_val, area_val):
    sim = ctrl.ControlSystemSimulation(capacity_ctrl)
    sim.input['confidence'] = float(conf_val)
    sim.input['area_ratio'] = float(area_val)
    try:
        sim.compute()
        score = sim.output['capacity']
    except Exception:
        score = (conf_val * 0.3 + area_val * 0.7) * 100
    
    if score >= 75:
        return round(score, 2), "PENUH", True
    elif score >= 45:
        return round(score, 2), "HAMPIR PENUH", False
    else:
        return round(score, 2), "AMAN", False

print("[INFO] Logika Fuzzy Terintegrasi.")

[INFO] Logika Fuzzy Terintegrasi.


3. Insialisasi Route Flask API

In [51]:
app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({'status': 'error', 'message': 'File gambar tidak ditemukan'}), 400
    
    file = request.files['file']
    img_bytes = file.read()
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    img_h, img_w, _ = img.shape
    total_area = img_h * img_w
    
    # =========================================================================
    # Diselaraskan: conf=0.40 agar responsif mendeteksi objek
    # =========================================================================
    results = model(img, conf=0.45)
    
    detections = []
    max_score = 0.0
    global_status = "AMAN"
    trigger_buzzer = False
    
    for result in results:
        for box in result.boxes:
            cls_id = int(box.cls[0].item())
            cls_name = model.names[cls_id]
            conf = float(box.conf[0].item())
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            box_area = (x2 - x1) * (y2 - y1)
            area_ratio_val = box_area / total_area
            
            # Filter tubuh/baju manusia (> 65% layar)
            if area_ratio_val > 0.65:
                continue
            
            score, status, buzzer = compute_fuzzy(conf, area_ratio_val)
            if score > max_score:
                max_score = score
                global_status = status
                trigger_buzzer = buzzer
                
            detections.append({
                'class_name': cls_name,
                'confidence': round(conf, 4),
                'area_ratio': round(area_ratio_val, 4),
                'fuzzy_capacity': score,
                'status': status
            })
            
    return jsonify({
        'status': 'success',
        'total_objects': len(detections),
        'system_summary': {
            'capacity_percentage': max_score,
            'container_status': global_status,
            'trigger_buzzer_sound': trigger_buzzer
        },
        'predictions': detections
    })

print("[INFO] Route /predict Siap Dijalankan (Conf 0.50 + Area Filter > 0.60).")

[INFO] Route /predict Siap Dijalankan (Conf 0.50 + Area Filter > 0.60).


4. Jalankan Server Flask API

In [52]:
import threading

# Fungsi untuk menjalankan Flask Server di latar belakang (background)
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# Jalankan server via threading agar Jupyter Notebook tidak terkunci/hang
if __name__ == '__main__':
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
    print("[INFO] Server Flask API Aktif di http://localhost:5000 (Background Thread)")

[INFO] Server Flask API Aktif di http://localhost:5000 (Background Thread)
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.211.79.152:5000
Press CTRL+C to quit


5. Pengujian Request ke Server API

In [53]:
import requests
from pathlib import Path

url = 'http://localhost:5000/predict'

# 1. Menentukan direktori folder test
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
test_dir = ROOT / "dataset" / "images" / "test"

# 2. Cari file gambar apa saja yang ada di folder test (.jpg, .png, .jpeg)
image_files = list(test_dir.glob("*.jpg")) + list(test_dir.glob("*.png")) + list(test_dir.glob("*.jpeg"))

if not image_files:
    print(f"[ERROR] Tidak ada file gambar ditemukan di direktori: {test_dir}")
else:
    # Ambil file gambar pertama yang ditemukan
    sample_image = image_files[0]
    print(f"[INFO] Menggunakan gambar sampel: {sample_image.name}")
    
    try:
        with open(sample_image, 'rb') as img_file:
            files = {'file': img_file}
            print(f"[INFO] Mengirim request ke {url}...")
            response = requests.post(url, files=files)
            
        print("\n=== HASIL RESPONSE DARI FLASK API ===")
        print("Status Code :", response.status_code)
        print("Response JSON:")
        import json
        print(json.dumps(response.json(), indent=4))
        
    except Exception as e:
        print("[ERROR] Gagal melakukan request ke API (Pastikan Cell 4 Server Flask sudah aktif):", e)

[INFO] Menggunakan gambar sampel: logam_004.jpg
[INFO] Mengirim request ke http://localhost:5000/predict...

0: 640x640 1 logam, 120.9ms
Speed: 7.4ms preprocess, 120.9ms inference, 3.6ms postprocess per image at shape (1, 3, 640, 640)


127.0.0.1 - - [10/Nov/2026 12:08:22] "POST /predict HTTP/1.1" 200 -



=== HASIL RESPONSE DARI FLASK API ===
Status Code : 200
Response JSON:
{
    "predictions": [
        {
            "area_ratio": 0.3137,
            "class_name": "logam",
            "confidence": 0.7869,
            "fuzzy_capacity": 38.76,
            "status": "AMAN"
        }
    ],
    "status": "success",
    "system_summary": {
        "capacity_percentage": 38.76,
        "container_status": "AMAN",
        "trigger_buzzer_sound": false
    },
    "total_objects": 1
}
